# Tensors and Forward Propagation

We will turn a small table of greenhouse readings into output scores using PyTorch tensors. The goal is to follow both the values and their shapes through a complete forward pass, then reproduce the result with a neural-network module.

A tensor is an array of numbers. We use rows for examples and columns for features. Our hand-set model combines readings, applies ReLU (keep positive values, replace negative values by zero), and combines the hidden values into one score per example. No training or performance evaluation occurs, and the scores are not validated watering advice.

Run all cells in order from a fresh kernel using the [README setup](README.md). Only PyTorch is needed for these CPU calculations; there are no downloads or credentials. The [notes](05_Tensors_and_Forward_Propagation_Notes.ipynb) provide optional conceptual background. All required values are defined here.

## Start with the problem this lesson solves

A network must process many examples efficiently while keeping each feature paired with the right weight. A tensor is a multidimensional array with named roles for its axes in our design. Forward propagation is the sequence of calculations that turns the input tensor into predictions using the current parameters.

The practical problem is not only calculating a number; it is calculating the intended number. Two arrays can have compatible shapes yet represent the wrong feature pairings or bias direction.

## Expand a batched forward pass into ordinary arithmetic

Rows of the input are greenhouses and columns are dryness and warmth:

$$X=\begin{bmatrix}0.8&0.4\\0.2&0.9\\0.5&0.5\end{bmatrix},\quad
W=\begin{bmatrix}1&1\\1&-1\\-1&1\end{bmatrix},\quad b=[0,-0.5,0.25].$$

$X$ has shape $(3,2)$ and $W$ has shape $(3,2)$ because each of three neurons has two weights. We use $Z=XW^T+b$: $(3,2)\times(2,3)$ produces a $(3,3)$ table. Each result is one greenhouse–neuron pairing.

| Greenhouse | Neuron one | Neuron two | Neuron three |
| --- | --- | --- | --- |
| A | $0.8+0.4+0=1.2$ | $0.8-0.4-0.5=-0.1$ | $-0.8+0.4+0.25=-0.15$ |
| B | $0.2+0.9+0=1.1$ | $0.2-0.9-0.5=-1.2$ | $-0.2+0.9+0.25=0.95$ |
| C | $0.5+0.5+0=1.0$ | $0.5-0.5-0.5=-0.5$ | $-0.5+0.5+0.25=0.25$ |

ReLU replaces every negative entry with zero:

$$H=\begin{bmatrix}1.2&0&0\\1.1&0&0.95\\1&0&0.25\end{bmatrix}.$$

The output uses weight row $[2,-1,0.5]$ and bias $-0.5$:

- A: $1.2(2)+0(-1)+0(0.5)-0.5=1.9$.
- B: $1.1(2)+0(-1)+0.95(0.5)-0.5=2.175$.
- C: $1(2)+0(-1)+0.25(0.5)-0.5=1.625$.

The final tensor has shape $(3,1)$: one score per greenhouse. These scores are not automatically probabilities or watering quantities; a task and output interpretation must define their meaning.

**Why bias direction matters.** Each neuron must use its own bias on every row. Before bias, A's three sums are $[1.2,0.4,-0.4]$. Adding $[0,-0.5,0.25]$ across that row gives the correct $[1.2,-0.1,-0.15]$. If we instead add a column vector, A receives zero on all three entries, B receives $-0.5$ on all entries, and C receives $0.25$ on all entries. The shape still happens to be $(3,3)$, but the model is different.

## Why tensors help the ANN do its job

Matrix multiplication performs the same multiply-and-add operation for many neurons and examples at once. It makes the computation efficient without changing its logic. Transpose aligns features with their corresponding weights; broadcasting reuses each bias on the intended axis.

A forward pass changes intermediate values, not stored parameters. To learn, we still need targets, a loss, gradients, and an optimizer. In this MLP, examples do not mix during prediction; layers such as training-mode batch normalization can behave differently, so this independence should not be assumed for every architecture.

## Connect the explanation to the code

`X @ W.T + b` is the expanded table above. `torch.relu` acts separately on each table entry. `H @ V.T + c` combines the hidden features into one score per row. `reshape` regroups stored entries; `.T` swaps the two axes. The bias demonstration shows why a shape check must be accompanied by a check of axis meaning and actual values.

The calculations above explain the mechanism before the full experiment. Read the following code cells in order; the printed values and assertions let you compare the implementation with its mathematical meaning.

In [1]:
import platform
import torch
from torch import nn

SEED = 42
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
torch.set_printoptions(precision=3, sci_mode=False)
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__)
print("CPU | fixed parameters, no training")

Python: 3.9.6 | PyTorch: 2.2.2
CPU | fixed parameters, no training


## Inspect the input contract

Each greenhouse has dryness and warmth readings on a scale from zero to one. `float32` retains fractional values, and `device="cpu"` makes the computation location explicit. Integer indexing removes an axis; slicing retains it. A scalar has no axes but still contains one value.

In [2]:
X = torch.tensor([[0.8, 0.4], [0.2, 0.9], [0.5, 0.5]],
                 dtype=torch.float32, device="cpu")
scalar = torch.tensor(0.8)
for name, value in [("scalar", scalar), ("one feature vector", X[0]),
                    ("one-row batch", X[0:1]), ("full batch", X)]:
    print(f"{name:20s} shape={tuple(value.shape)} axes={value.ndim} values={value.numel()}")
print("Input dtype:", X.dtype, "| device:", X.device)
print("Columns: dryness, warmth\n", X)
assert X.shape == (3, 2)
assert X[0:1].shape == (1, 2)

scalar               shape=() axes=0 values=1
one feature vector   shape=(2,) axes=1 values=2
one-row batch        shape=(1, 2) axes=2 values=2
full batch           shape=(3, 2) axes=2 values=6
Input dtype: torch.float32 | device: cpu
Columns: dryness, warmth
 tensor([[0.800, 0.400],
        [0.200, 0.900],
        [0.500, 0.500]])


## Separate contributions from a weighted sum

The weights `(1, -1)` compare dryness with warmth. `*` keeps each individual contribution; summing those contributions produces the dot product. We check the equivalence numerically before using matrix multiplication for a whole layer.

In [3]:
comparison_weights = torch.tensor([1.0, -1.0])
contributions = X[0] * comparison_weights
weighted_sum = contributions.sum()
print("First-row contributions:", contributions)
print("Their sum:", weighted_sum.item())
print("Dot product:", (X[0] @ comparison_weights).item())
torch.testing.assert_close(weighted_sum, X[0] @ comparison_weights)

First-row contributions: tensor([ 0.800, -0.400])
Their sum: 0.4000000059604645
Dot product: 0.4000000059604645


## Compute the hidden layer explicitly

Each row of `W` stores one hidden unit's weights. `W.T` transposes the matrix, so `(batch, features) @ (features, units)` gives `(batch, units)`. The vector `b` supplies one bias per hidden unit, shared across examples. We compare all scores and activated values with hand-calculated expected matrices.

In [4]:
W = torch.tensor([[1., 1.], [1., -1.], [-1., 1.]])
b = torch.tensor([0., -0.5, 0.25])
Z = X @ W.T + b
H = torch.relu(Z)
expected_Z = torch.tensor([[1.2, -0.1, -0.15],
                           [1.1, -1.2, 0.95],
                           [1.0, -0.5, 0.25]])
expected_H = torch.tensor([[1.2, 0., 0.], [1.1, 0., 0.95], [1., 0., 0.25]])
torch.testing.assert_close(Z, expected_Z)
torch.testing.assert_close(H, expected_H)
print("X", tuple(X.shape), "@ W.T", tuple(W.T.shape), "+ b", tuple(b.shape))
print("Hidden scores:\n", Z)
print("After ReLU:\n", H)

X (3, 2) @ W.T (2, 3) + b (3,)
Hidden scores:
 tensor([[ 1.200, -0.100, -0.150],
        [ 1.100, -1.200,  0.950],
        [ 1.000, -0.500,  0.250]])
After ReLU:
 tensor([[1.200, 0.000, 0.000],
        [1.100, 0.000, 0.950],
        [1.000, 0.000, 0.250]])


## Catch a broadcast that runs but means the wrong thing

Broadcasting compares dimensions from the right: they must match or one must be one. A row bias `(1, 3)` repeats across examples. A column `(3, 1)` repeats across units and assigns offsets by example instead. Both run here because the batch size happens to equal the hidden width. We verify that the latter calculation is wrong for our intended model.

In [5]:
base_scores = X @ W.T
correct = base_scores + b.reshape(1, -1)
wrong = base_scores + b.reshape(-1, 1)
print("Correct per-unit bias:\n", correct)
print("Incorrect per-example offsets:\n", wrong)
torch.testing.assert_close(correct, Z)
assert not torch.allclose(wrong, Z)
print("Same output shape:", correct.shape == wrong.shape, "| same values: False")

Correct per-unit bias:
 tensor([[ 1.200, -0.100, -0.150],
        [ 1.100, -1.200,  0.950],
        [ 1.000, -0.500,  0.250]])
Incorrect per-example offsets:
 tensor([[ 1.200,  0.400, -0.400],
        [ 0.600, -1.200,  0.200],
        [ 1.250,  0.250,  0.250]])
Same output shape: True | same values: False


## Finish the forward pass

The output layer multiplies each hidden row by `(2, -1, 0.5)` and adds `-0.5`. The expected scores are `1.9`, `2.175`, and `1.625`. The final singleton axis means one score per example; it is intentionally retained. No probability transformation is appropriate for interpreting these invented parameters as real predictions.

In [6]:
V = torch.tensor([[2., -1., 0.5]])
c = torch.tensor([-0.5])
S = H @ V.T + c
expected_S = torch.tensor([[1.9], [2.175], [1.625]])
torch.testing.assert_close(S, expected_S)
print("Hidden", tuple(H.shape), "-> output", tuple(S.shape))
print("Greenhouse   dryness   warmth   score")
for label, readings, score in zip(["A", "B", "C"], X, S):
    print(f"{label:10s} {readings[0].item():8.2f} {readings[1].item():8.2f} {score.item():7.3f}")

Hidden (3, 3) -> output (3, 1)
Greenhouse   dryness   warmth   score
A              0.80     0.40   1.900
B              0.20     0.90   2.175
C              0.50     0.50   1.625


## Reproduce the calculation with a model

`nn.Linear` includes the matrix multiplication and bias addition; it does not include ReLU. We copy our chosen parameters into the layers inside `torch.no_grad()`, which avoids recording these setup operations for gradients. `model.eval()` selects evaluation behavior; it does not modify this dense/ReLU computation or learn weights.

In [7]:
model = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Linear(3, 1))
with torch.no_grad():
    model[0].weight.copy_(W)
    model[0].bias.copy_(b)
    model[2].weight.copy_(V)
    model[2].bias.copy_(c)
model.eval()
with torch.no_grad():
    module_scores = model(X)
torch.testing.assert_close(module_scores, S)
print(model)
print("Explicit tensors and nn.Sequential agree:\n", module_scores)

Sequential(
  (0): Linear(in_features=2, out_features=3, bias=True)
  (1): ReLU()
  (2): Linear(in_features=3, out_features=1, bias=True)
)
Explicit tensors and nn.Sequential agree:
 tensor([[1.900],
        [2.175],
        [1.625]])


## Check what batch independence means

For this model, each example follows its own path through shared parameters. We check that batching agrees with processing one row at a time, reordering inputs reorders outputs, and changing one row leaves the other rows unchanged. These checks would need reconsideration for a model that mixes information across examples.

We also snapshot parameters before the forward calls and verify that the calls do not update them.

In [8]:
parameters_before = {name: value.detach().clone() for name, value in model.named_parameters()}
order = torch.tensor([2, 0, 1])
changed_X = X.clone()
changed_X[0] = torch.tensor([0., 0.])
with torch.no_grad():
    rowwise = torch.cat([model(row.unsqueeze(0)) for row in X], dim=0)
    reordered = model(X[order])
    changed_scores = model(changed_X)
torch.testing.assert_close(rowwise, module_scores)
torch.testing.assert_close(reordered, module_scores[order])
torch.testing.assert_close(changed_scores[1:], module_scores[1:])
assert not torch.allclose(changed_scores[0], module_scores[0])
for name, value in model.named_parameters():
    torch.testing.assert_close(value, parameters_before[name])
print("Batched and rowwise outputs agree.")
print("Reordered outputs:", reordered.flatten().tolist())
print("Changing only greenhouse A gives:\n", changed_scores)
print("All stored parameters are unchanged.")

Batched and rowwise outputs agree.
Reordered outputs: [1.625, 1.9000000953674316, 2.174999952316284]
Changing only greenhouse A gives:
 tensor([[-0.375],
        [ 2.175],
        [ 1.625]])
All stored parameters are unchanged.


## Distinguish rearranging axes from reshaping

The following integer table is only an arrangement demonstration, not input to our floating-point model. Transposing exchanges row and column axes. Reshaping groups values in their existing traversal order. Matching shapes therefore do not establish matching feature assignments.

In [9]:
table = torch.tensor([[1, 2, 3], [4, 5, 6]])
transposed = table.T
reshaped = table.reshape(3, 2)
print("Original:\n", table)
print("Transpose:\n", transposed)
print("Reshape:\n", reshaped)
assert transposed.shape == reshaped.shape
assert not torch.equal(transposed, reshaped)
print("Same shape, different pairings.")

Original:
 tensor([[1, 2, 3],
        [4, 5, 6]])
Transpose:
 tensor([[1, 4],
        [2, 5],
        [3, 6]])
Reshape:
 tensor([[1, 2],
        [3, 4],
        [5, 6]])
Same shape, different pairings.


## Interpret the results

The explicit tensor computation and the model produced identical scores within floating-point tolerance. Each assertion checks a meaningful property: agreement with worked arithmetic, correct bias alignment, equivalent implementations, or preservation of example identity. This establishes that the forward pass implements our intended calculation; it makes no claim about usefulness on real greenhouse data.

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

numeric_train = [[2., 100.], [4., 200.], [6., 300.]]
numeric_scaler = MinMaxScaler().fit(numeric_train)
numeric_test = numeric_scaler.transform([[3., 150.], [8., 350.]])
assert np.allclose(numeric_test[0], [.25, .25])
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit([['small'], ['large']])
categorical = encoder.transform([['large'], ['small'], ['medium']])
assert np.array_equal(categorical[-1], [0., 0.])
print('scaled:', numeric_test.tolist(), '| encoded:', categorical.tolist())